# Scikit-Learn (sklearn) - Machine Learning

**What:** The main Python library for machine learning. Provides ready-to-use algorithms for classification, regression, clustering, and data preprocessing.

**When to use:**
- **Classification:** Predict categories (spam/not spam, promoted/not promoted)
- **Regression:** Predict numbers (salary, price, temperature)
- **Clustering:** Group similar data points (customer segments)
- **Preprocessing:** Scale features, encode labels, handle missing data
- **Model evaluation:** Accuracy, confusion matrix, cross-validation
- **Dimensionality reduction:** PCA to reduce features

**Key concept:** All models follow the same pattern: `model.fit(X_train, y_train)` → `model.predict(X_test)` → `accuracy_score()`

In [ ]:
# Install dependencies (run once)
!uv add scikit-learn pandas numpy matplotlib

In [ ]:
# 1. Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

print("Sklearn ready!")

In [ ]:
# 2. Create Sample Dataset
np.random.seed(42)
n = 200

df = pd.DataFrame({
    "age": np.random.randint(20, 60, n),
    "experience": np.random.randint(0, 30, n),
    "education_level": np.random.choice(["High School", "Bachelor", "Master", "PhD"], n),
    "salary": np.random.normal(55000, 15000, n).astype(int),
    "performance": np.random.uniform(1.0, 5.0, n).round(1),
    "promoted": np.random.choice([0, 1], n, p=[0.7, 0.3]),
})

print(df.head(10))
print(f"\nShape: {df.shape}")

In [ ]:
# 3. Label Encoding (Text to Numbers)
encoder = LabelEncoder()
df["education_encoded"] = encoder.fit_transform(df["education_level"])

print("Mapping:")
for label, code in zip(encoder.classes_, range(len(encoder.classes_))):
    print(f"  {label} -> {code}")

print(f"\n{df[['education_level', 'education_encoded']].head()}")

In [ ]:
# 4. Train-Test Split
X = df[["age", "experience", "education_encoded", "salary", "performance"]]
y = df["promoted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

In [ ]:
# 5. Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Before scaling:")
print(f"  Mean: {X_train['salary'].mean():.0f}, Std: {X_train['salary'].std():.0f}")
print(f"\nAfter scaling:")
print(f"  Mean: {X_train_scaled[:, 3].mean():.2f}, Std: {X_train_scaled[:, 3].std():.2f}")

## Classification Algorithms

In [ ]:
# 6. Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

print("Logistic Regression")
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.2%}")
print(f"\n{classification_report(y_test, lr_pred)}")

In [ ]:
# 7. Decision Tree
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

print("Decision Tree")
print(f"Accuracy: {accuracy_score(y_test, dt_pred):.2%}")

# Feature importance
importance = pd.Series(dt_model.feature_importances_, index=X.columns)
print(f"\nFeature Importance:")
print(importance.sort_values(ascending=False))

In [ ]:
# 8. Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

print("Random Forest")
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.2%}")

# Feature importance
importance = pd.Series(rf_model.feature_importances_, index=X.columns)
plt.figure(figsize=(8, 4))
importance.sort_values().plot(kind="barh", color="#4CAF50")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.show()

In [ ]:
# 9. Support Vector Machine (SVM)
from sklearn.svm import SVC

svm_model = SVC(kernel="rbf", random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)

print("SVM")
print(f"Accuracy: {accuracy_score(y_test, svm_pred):.2%}")

In [ ]:
# 10. K-Nearest Neighbors (KNN)
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)

print("KNN (k=5)")
print(f"Accuracy: {accuracy_score(y_test, knn_pred):.2%}")

In [ ]:
# 11. Compare All Classifiers
models = {
    "Logistic Regression": accuracy_score(y_test, lr_pred),
    "Decision Tree": accuracy_score(y_test, dt_pred),
    "Random Forest": accuracy_score(y_test, rf_pred),
    "SVM": accuracy_score(y_test, svm_pred),
    "KNN": accuracy_score(y_test, knn_pred),
}

comparison = pd.DataFrame(list(models.items()), columns=["Model", "Accuracy"])
comparison = comparison.sort_values("Accuracy", ascending=False)
print(comparison.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.barh(comparison["Model"], comparison["Accuracy"], color="#2196F3")
plt.title("Model Accuracy Comparison")
plt.xlabel("Accuracy")
plt.xlim(0, 1)
plt.show()

## Regression Algorithms

In [ ]:
# 12. Linear Regression
from sklearn.linear_model import LinearRegression

# Predict salary based on age and experience
X_reg = df[["age", "experience", "education_encoded", "performance"]]
y_reg = df["salary"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

lin_model = LinearRegression()
lin_model.fit(X_train_r, y_train_r)
lin_pred = lin_model.predict(X_test_r)

print("Linear Regression")
print(f"R2 Score: {r2_score(y_test_r, lin_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, lin_pred)):.0f}")
print(f"\nCoefficients:")
for feat, coef in zip(X_reg.columns, lin_model.coef_):
    print(f"  {feat}: {coef:.2f}")

In [ ]:
# 13. Actual vs Predicted Plot
plt.figure(figsize=(8, 5))
plt.scatter(y_test_r, lin_pred, alpha=0.5, color="#2196F3")
plt.plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], "r--", linewidth=2)
plt.title("Actual vs Predicted Salary")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.grid(True, alpha=0.3)
plt.show()

## Clustering Algorithms

In [ ]:
# 14. K-Means Clustering
from sklearn.cluster import KMeans

# Cluster employees by salary and experience
X_cluster = df[["salary", "experience"]].values
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_cluster_scaled)

plt.figure(figsize=(8, 5))
for cluster in range(3):
    mask = df["cluster"] == cluster
    plt.scatter(df[mask]["experience"], df[mask]["salary"], label=f"Cluster {cluster}", alpha=0.6)

plt.title("K-Means Clustering (k=3)")
plt.xlabel("Experience")
plt.ylabel("Salary")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 15. Elbow Method (Find Optimal K)
inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, marker="o", color="#F44336")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.grid(True, alpha=0.3)
plt.show()

## Extras

In [ ]:
# 16. PCA (Dimensionality Reduction)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)

print(f"Original features: {X_train_scaled.shape[1]}")
print(f"Reduced features: {X_pca.shape[1]}")
print(f"Variance explained: {pca.explained_variance_ratio_}")
print(f"Total variance: {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap="viridis", alpha=0.6)
plt.colorbar(label="Promoted")
plt.title("PCA Visualization (2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
# 17. Cross Validation
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring="accuracy")

print("5-Fold Cross Validation (Random Forest)")
print(f"Scores: {scores}")
print(f"Mean:   {scores.mean():.2%}")
print(f"Std:    {scores.std():.2%}")

In [ ]:
# 18. Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Not Promoted", "Promoted"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix (Random Forest)")
plt.show()

In [ ]:
# 19. Save and Load Model
import joblib

# Save
joblib.dump(rf_model, "random_forest_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Model saved!")

# Load
loaded_model = joblib.load("random_forest_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")
print("Model loaded!")

# Predict with loaded model
sample = np.array([[35, 10, 2, 60000, 4.0]])
sample_scaled = loaded_scaler.transform(sample)
prediction = loaded_model.predict(sample_scaled)
print(f"\nPrediction: {'Promoted' if prediction[0] == 1 else 'Not Promoted'}")